# RE_203 · A qui truquem?

**Bloc 2 · Supervisat · repte 3 de 6 · durada: 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Ets analista al departament de màrqueting d'un banc. La propera campanya ofereix un **dipòsit a
termini** per telèfon. Cada trucada costa temps i diners, i la majoria de clients diuen que no. La
directora vol una llista de clients a qui trucar, i et pregunta:

**un model pot triar a qui trucar millor que l'atzar? Amb quina mesura s'ha de jutjar, i on poses el
llindar per decidir si truques o no?**

L'enunciat complet, amb el diccionari de columnes, és a [RE_203](../RE_203_campanya_banc.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- Matriu de confusió, *precision*, *recall*, F1 i `DummyClassifier`: [EX_00 §7](../../02_practica/EX_00_caixa_eines.ipynb)
- Moure el llindar de decisió: [EX_02 exercicis 5-7](../../02_practica/EX_02_cancer.ipynb)
- Boscos aleatoris i per què la precisió sola no serveix: [ML_03](../../01_teoria/ML_03_boscos.ipynb)
- Regressió logística i `predict_proba`: [ML_04](../../01_teoria/ML_04_regressio_logistica.ipynb)
- `ColumnTransformer` amb numèriques i categòriques: [EX_00 §6](../../02_practica/EX_00_caixa_eines.ipynb)

## Les dades

**Bank Marketing** (Moro, Cortez i Rita, 2014; UCI), a OpenML amb `data_id=46910`. Llicència CC BY
4.0. Cada fila és **un client** d'un banc portuguès contactat en una campanya telefònica (2008-2010):
edat, feina, estat civil, estudis, saldo, si té hipoteca o préstec, com se'l va contactar, quantes
vegades en aquesta campanya i en l'anterior... La columna `SubscribeTermDeposit` diu si va contractar
el dipòsit. No hi ha cap identificador de persona.

Aquesta versió **ja no porta** la durada de la trucada, que en la versió original revelava el
resultat (no se sap quant durarà una trucada abans de fer-la). Documentació:
<https://archive.ics.uci.edu/dataset/222/bank+marketing>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=46910, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Quants diuen que sí

Forma, tipus i proporció de clients que contracten. Converteix l'objectiu en `y` de 0 i 1.

In [ ]:
print(df.dtypes)
print(df["SubscribeTermDeposit"].value_counts(normalize=True).round(4))
y = (df["SubscribeTermDeposit"] == "yes").astype(int)
assert len(y) == 45211 and set(y.unique()) == {0, 1}
print("Proporció de sí:", round(y.mean(), 4))

> **Resultat.** 11,7 % de «sí» (5.289 de 45.211). És el número que s'ha de recordar tot el repte.

### 1.2 · `pdays = -1` i `unknown`

Quantes files tenen `pdays == -1`? Mira la documentació i les columnes `previous` i `poutcome` d'aquestes files: què vol dir? Després compta quants `unknown` hi ha a cada columna de text. Per a cada columna, decideix si `unknown` és un nul o una categoria amb significat.

In [ ]:
menys1 = df["pdays"] == -1
print("pdays == -1:", menys1.sum(), f"({menys1.mean():.1%})")
print("previous màxim en aquestes files:", df.loc[menys1, "previous"].max())
print(df.loc[menys1, "poutcome"].value_counts())
cols_text = df.select_dtypes(exclude="number").columns
print((df[cols_text] == "unknown").sum().sort_values(ascending=False))

> **Resultat.** 36.954 files (81,7 %) tenen `pdays = -1`: són clients **mai contactats abans** (`previous` val 0 i `poutcome` és `unknown`). No és un nul, és una situació. `unknown` és a `poutcome` (36.959), `contact` (13.020), `education` (1.857) i `job` (288). A `poutcome` vol dir «no hi va haver campanya anterior»; a `job` i `education` sí que és un nul. Una decisió raonable és deixar-lo com a categoria pertot i dir-ho.

### 1.3 · Duplicats

Quantes files duplicades hi ha? Sense identificador de client, és possible distingir un error de dos clients amb el mateix perfil? Decideix què en fas.

In [ ]:
print("Duplicats:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(6)

> **Resultat.** 274 duplicats sobre 45.211 (0,6 %). Sense identificador, poden ser clients diferents amb el mateix perfil. Aquí es deixen: canvien molt poc res i esborrar-los també és una decisió defensable.

## Nivell 2 · Respondre

### 2.1 · El model que no truca mai

Parteix (25 % de test, estratificat, `random_state=42`). Calcula la precisió, el *recall* i la *precision* d'un `DummyClassifier` que sempre diu «no». Per què és un mal model encara que la precisió sigui alta?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X = df.drop(columns="SubscribeTermDeposit")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
pred_dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train).predict(X_test)
print("accuracy:", round(accuracy_score(y_test, pred_dummy), 4),
      "| recall:", recall_score(y_test, pred_dummy),
      "| precision:", precision_score(y_test, pred_dummy, zero_division=0))
assert recall_score(y_test, pred_dummy) == 0

> **Resultat.** Precisió de 0,883 sense trucar ningú: la precisió premia dir «no» i aquí el negoci és trobar els «sí». *Recall* 0.

### 2.2 · Dos models i les seves matrius

Prepara un `ColumnTransformer` (escalat de numèriques i one-hot de categòriques) i entrena una **regressió logística** i un **bosc aleatori**, cadascun dins d'un `Pipeline`. Treu la matriu de confusió de test de cadascun, amb el llindar per defecte.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

num = list(X.select_dtypes("number").columns)
cat = [c for c in X.columns if c not in num]
pre = ColumnTransformer([("num", StandardScaler(), num),
                         ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
models = {
    "logística": make_pipeline(pre, LogisticRegression(max_iter=1000)),
    "bosc": make_pipeline(pre, RandomForestClassifier(n_estimators=200, min_samples_leaf=5,
                                                      random_state=42, n_jobs=-1)),
}
cms, probs = {}, {}
for nom, m in models.items():
    m.fit(X_train, y_train)
    probs[nom] = m.predict_proba(X_test)[:, 1]
    cms[nom] = confusion_matrix(y_test, m.predict(X_test))
    assert cms[nom].sum() == len(y_test) and cms[nom][1].sum() == y_test.sum()
    print(nom, "\n", cms[nom])

> **Resultat.** Amb el llindar de 0,5 tots dos models diuen «sí» a molt poca gent: la logística en troba 233 dels 1.322 «sí» de test i el bosc 232 (amb 330 trucades cadascun). Encara són molt a prop del model que no truca mai.

### 2.3 · La teoria, al codi: les mesures des de la matriu

Escriu una funció que, a partir **només** de les quatre caselles d'una matriu de confusió, retorni *accuracy*, *precision*, *recall* i F1. Aplica-la a les dues matrius i comprova cada número contra la funció de scikit-learn corresponent. Quin model és millor segons cada mesura?

In [ ]:
def mesures(cm):
    tn, fp, fn, tp = cm.ravel()
    acc = (tp + tn) / (tn + fp + fn + tp)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn)
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1)

taula = {}
for nom, m in models.items():
    pred = m.predict(X_test)
    meu = mesures(cms[nom])
    sk = dict(accuracy=accuracy_score(y_test, pred), precision=precision_score(y_test, pred),
              recall=recall_score(y_test, pred), f1=f1_score(y_test, pred))
    assert all(np.isclose(meu[k], sk[k]) for k in meu)
    taula[nom] = meu
pd.DataFrame(taula).round(4)

> **Resultat.** Coincideixen tots. Amb el llindar de 0,5 els dos models empaten pràcticament (accuracy 0,895, precision 0,70, *recall* 0,18, F1 0,28 tots dos): el llindar per defecte no deixa veure cap diferència, i el *recall* és baixíssim. La diferència apareix a l'2.5, quan es mira tota la corba.

### 2.4 · On poses el llindar

Amb les probabilitats del millor model, prova llindars de 0,05 a 0,90 i fes una taula amb *precision*, *recall*, F1 i **nombre de trucades**. Ara un escenari (inventat, però raonable): cada trucada costa **5 €** i cada client que contracta deixa **50 €** de marge. Afegeix a la taula el **benefici** de cada llindar, i compara'l amb trucar a tothom. Quin llindar triaries?

In [ ]:
p = probs["bosc"]
files = []
for ll in [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]:
    pred = (p >= ll).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    m = mesures(np.array([[tn, fp], [fn, tp]]))
    files.append(dict(llindar=ll, trucades=tp + fp, precision=m["precision"], recall=m["recall"],
                      f1=m["f1"], benefici=50 * tp - 5 * (tp + fp)))
t = pd.DataFrame(files).set_index("llindar")
assert t.loc[0.0, "recall"] == 1 and t["trucades"].is_monotonic_decreasing
print(t.round(3))
print("Trucar a tothom:", 50 * y_test.sum() - 5 * len(y_test), "€")

> **Resultat.** Amb el bosc, el benefici màxim surt amb llindar 0,15 (2.396 trucades, *recall* 0,56, 25.120 €), molt per sobre de trucar a tothom (9.585 €) i del llindar de 0,5 (9.950 €, perquè gairebé no truca). Fixeu-vos que el millor F1 (0,20) no és el millor benefici (0,15). El llindar òptim depèn dels costos: si canvien els 5 € i els 50 €, canvia. Les xifres exactes són a la taula de sortida.

### 2.5 · ROC o precision-recall?

Dibuixa la corba ROC i la corba *precision-recall* dels dos models, i calcula'n les àrees. Fes el mateix amb un «model» que dona probabilitats a l'atzar. Quina de les dues corbes explica millor aquest problema a la directora?

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, RocCurveDisplay, PrecisionRecallDisplay

atzar = np.random.default_rng(0).random(len(y_test))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for nom, pp in {**probs, "atzar": atzar}.items():
    RocCurveDisplay.from_predictions(y_test, pp, ax=ax[0], name=nom)
    PrecisionRecallDisplay.from_predictions(y_test, pp, ax=ax[1], name=nom)
    print(f"{nom:10s} ROC-AUC {roc_auc_score(y_test, pp):.3f} | AP {average_precision_score(y_test, pp):.3f}")
plt.show()
assert abs(roc_auc_score(y_test, atzar) - 0.5) < 0.03
assert abs(average_precision_score(y_test, atzar) - y_test.mean()) < 0.03

> **Resultat.** ROC-AUC 0,755 (logística) i 0,762 (bosc); AP 0,380 i 0,413; l'atzar fa 0,522 i 0,124 (≈ la proporció de «sí»). L'àrea ROC sembla bona i amaga que encara hi ha molts falsos positius; l'AP ho ensenya, i és on el bosc es despega una mica. Com que la classe interessant és petita, la corba *precision-recall* és la que parla el llenguatge de la directora (de cada 10 trucades, quantes són «sí»).

## Nivell 3 · Més enllà

### 3.1 · `class_weight` o llindar?

Entrena la logística amb `class_weight="balanced"`. Compara la seva matriu de confusió (llindar 0,5) amb la de la logística normal amb un llindar més baix. Són dues maneres de fer el mateix? Mira també què passa amb la corba ROC.

In [ ]:
bal = make_pipeline(pre, LogisticRegression(max_iter=1000, class_weight="balanced")).fit(X_train, y_train)
pb = bal.predict_proba(X_test)[:, 1]
n_truc = int((pb >= 0.5).sum())
pl = probs["logística"]
ll_eq = np.sort(pl)[::-1][n_truc - 1]            # llindar que fa les mateixes trucades
print("balancejada (0,5):\n", confusion_matrix(y_test, (pb >= 0.5).astype(int)))
print(f"normal amb llindar {ll_eq:.3f}:\n", confusion_matrix(y_test, (pl >= ll_eq).astype(int)))
print("ROC-AUC normal:", round(roc_auc_score(y_test, pl), 4), "| balancejada:", round(roc_auc_score(y_test, pb), 4))

> **Resultat.** Amb el mateix nombre de trucades, les dues matrius són gairebé iguals i l'àrea ROC pràcticament no canvia: `class_weight` sobretot **desplaça** les probabilitats (i deixen de ser probabilitats calibrades), mentre que moure el llindar les deixa intactes.

### 3.2 · `pdays` com cal

El -1 de `pdays` fa que el model vegi «menys un dia». Crea una columna nova que digui si el client havia estat contactat abans, i substitueix el -1 per un valor que tingui sentit (o fes-ne una altra cosa que puguis defensar). Canvia alguna cosa en el millor model?

In [ ]:
d2 = df.copy()
d2["contactat_abans"] = (d2["pdays"] != -1).astype(int)
d2["pdays"] = d2["pdays"].replace(-1, 999)
X2 = d2.drop(columns="SubscribeTermDeposit")
X2tr, X2te, _, _ = train_test_split(X2, y, test_size=0.25, stratify=y, random_state=42)
num2 = list(X2.select_dtypes("number").columns)
pre2 = ColumnTransformer([("num", StandardScaler(), num2),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
for nom, est in [("logística", LogisticRegression(max_iter=1000)),
                 ("bosc", RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=42, n_jobs=-1))]:
    m2 = make_pipeline(pre2, est).fit(X2tr, y_train)
    print(nom, "AP abans:", round(average_precision_score(y_test, probs[nom]), 4),
          "| després:", round(average_precision_score(y_test, m2.predict_proba(X2te)[:, 1]), 4))

> **Resultat.** No canvia res (AP 0,3798 → 0,3797 i 0,4129 → 0,4120). La raó és bonica: el one-hot de `poutcome = unknown` ja era exactament l'indicador «mai contactat», i el model ja el tenia.

## Conclusió

Només un 11,7 % dels clients contracta, i per això la precisió no serveix: el model que no truca
mai ja en fa 0,883. Amb el llindar de 0,5, la logística i el bosc empaten (F1 0,28, *recall* 0,18) i
gairebé no truquen ningú. La diferència surt a la corba *precision-recall* (AP 0,413 el bosc i
0,380 la logística; l'atzar, 0,124), que és la que parla el llenguatge de la directora. Amb 5 € per
trucada i 50 € per contracte, el benefici màxim surt amb un llindar de 0,15: unes 2.400 trucades,
més de la meitat dels «sí» trobats i uns 25.000 € al test, contra uns 9.600 € trucant a tothom. Si
canvien els costos, canvia el llindar, no el model.